# Guide intelligent du Musée · 02 Baseline (jalon `m1`)

Deux modèles de référence pour reconnaître le style d'une œuvre : une régression logistique, le modèle classique à battre, puis un réseau dense écrit avec les briques de PyTorch.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import platform
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from torch import nn
from torch.utils.data import DataLoader

from commun import (MODELS_DIR, SEED, ArtworkDataset, DenseNet, evaluate, fit, get_device, load_images,
                    load_metadata, normalization_stats, predict, set_seed, split_indices, standardize)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Régression logistique de référence

Le modèle classique à battre : une régression logistique multinomiale sur les pixels aplatis (3 × 128 × 128 = 49 152 valeurs), standardisés avec les statistiques du train. Avec 49 152 variables pour 900 œuvres d'entraînement, le modèle peut apprendre le train par cœur : la force de la régularisation, `C` (plus il est petit, plus elle est forte), est choisie sur la validation. Le test n'est pas utilisé.

In [ ]:
def flat_pixels(idx: np.ndarray) -> np.ndarray:
    """Œuvres d'un split, standardisées puis aplaties en vecteurs de 49 152 valeurs."""
    return standardize(X_all[idx], MEAN, STD).flatten(1).numpy()


X_train, X_val = flat_pixels(splits["train"]), flat_pixels(splits["val"])
y_train, y_val = y_all[splits["train"]].numpy(), y_all[splits["val"]].numpy()
print(f"X_train : {X_train.shape}, X_val : {X_val.shape}")

### 2.1 Choix de la régularisation

Une valeur de `C` par ordre de grandeur, resserrée autour de l'optimum.

In [ ]:
C_GRID = [1e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 1e-3, 1e-2]
MAX_ITER = 2000

rows = []
for C in C_GRID:
    t0 = time.perf_counter()
    clf = LogisticRegression(C=C, max_iter=MAX_ITER).fit(X_train, y_train)
    assert clf.n_iter_[0] < MAX_ITER, f"C={C} : pas de convergence"
    rows.append({"C": C, "acc_train": clf.score(X_train, y_train), "acc_val": clf.score(X_val, y_val),
                 "entrainement_s": time.perf_counter() - t0})

grid = pd.DataFrame(rows).set_index("C")
grid.rename(index="{:g}".format).round(3)

### 2.2 Modèle retenu

En cas d'égalité sur la validation, la régularisation la plus forte l'emporte. Le temps d'inférence est mesuré image par image, comme sur une borne, prétraitement compris.

In [ ]:
best_C = grid["acc_val"].idxmax()  # premier maximum = plus petit C, la grille étant croissante

t0 = time.perf_counter()
logreg = LogisticRegression(C=best_C, max_iter=MAX_ITER).fit(X_train, y_train)
fit_s = time.perf_counter() - t0

acc_val = logreg.score(X_val, y_val)
half_ci = 1.96 * np.sqrt(acc_val * (1 - acc_val) / len(y_val))  # intervalle à 95 % sur 300 œuvres


def predict_one(x: torch.Tensor) -> int:
    """Style prédit pour une seule œuvre uint8 (C, H, W), prétraitement compris."""
    return int(logreg.predict(standardize(x[None], MEAN, STD).flatten(1).numpy())[0])


x = X_all[splits["val"][0]]
predict_one(x)  # premier appel hors mesure
latencies = []
for _ in range(200):
    t0 = time.perf_counter()
    predict_one(x)
    latencies.append(time.perf_counter() - t0)

print(f"C retenu : {best_C:g}")
print(f"Accuracy de validation : {acc_val:.3f} ({int(round(acc_val * len(y_val)))}/{len(y_val)}, ± {half_ci:.3f} à 95 %), "
      f"hasard : {1 / len(CLASSES):.2f}")
print(f"Entraînement : {fit_s:.1f} s ; inférence : {1000 * np.median(latencies):.1f} ms par image (médiane sur 200)")
print(f"Mesuré sur CPU : {platform.platform()}")

### 2.3 Sauvegarde

Le modèle est rechargé par les notebooks du ResNet et de la recommandation. Il est sauvegardé dans `modeles/`, qui n'est pas versionné.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
logreg_path = MODELS_DIR / "regression_logistique.joblib"
joblib.dump(logreg, logreg_path)

reloaded = joblib.load(logreg_path)
assert (reloaded.predict(X_val) == logreg.predict(X_val)).all()
print(f"{logreg_path.name} : {logreg_path.stat().st_size / 1e6:.1f} Mo, prédictions identiques après rechargement")

### 2.4 Bilan

- **Chiffre à battre : 0,427 d'accuracy de validation** (128 œuvres sur 300, avec C = 1e-5), contre 0,20 au hasard. Sur 300 œuvres, l'intervalle à 95 % fait environ ± 5,6 points : un modèle ne sera dit meilleur que s'il dépasse nettement cet écart.
- **Le réglage de `C` est décisif.** Trop régularisé (C = 1e-7), le modèle sous-apprend : 0,331 en train, 0,313 en validation. Peu régularisé, il apprend le train par cœur : dès C = 1e-3, 0,988 en train mais 0,343 en validation. Avec 49 152 pixels pour 900 œuvres, c'est le sur-apprentissage attendu.
- **Un coût négligeable pour une borne** : quelques secondes d'entraînement, moins d'une milliseconde par image et un modèle de 1,0 Mo. Ces temps sont mesurés sur un ordinateur portable récent ; une borne sera plus lente, mais la marge est large.
- **Sa limite** : il ne combine les pixels que de façon linéaire, chacun à une position fixe. Le réseau dense ajoute des combinaisons non linéaires ; seul le ResNet, convolutif, reconnaîtra un motif où qu'il soit dans l'image.

## 3. Réseau dense de référence

Un réseau feed-forward écrit avec les briques de PyTorch, dans l'esprit des séances 4 et 5 : les pixels aplatis, une couche cachée de 128 neurones en sigmoïde, puis 5 sorties (les logits), entraîné par SGD avec la boucle propre de `commun.py` (forward, perte, `zero_grad`, `backward`, `step` ; `eval()` et `no_grad()` pour évaluer). ReLU, l'initialisation de He, Adam et la régularisation sont les corrections testées au jalon `m2`.

Le réseau est entraîné sur CPU : il est petit, et les résultats restent reproductibles d'une exécution à l'autre. Le test n'est pas utilisé.

### 3.1 Données et réseau

In [ ]:
N_IN = 3 * 128 * 128
HIDDEN = [128]
BATCH_SIZE = 64
CPU = torch.device("cpu")

datasets = {k: ArtworkDataset(X_all, y_all, splits[k], MEAN, STD) for k in ("train", "val")}
val_loader = DataLoader(datasets["val"], batch_size=256)


def make_train_loader() -> DataLoader:
    return DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)


set_seed()
model = DenseNet(N_IN, HIDDEN, len(CLASSES))
print(model)
print(f"{sum(p.numel() for p in model.parameters()):_} paramètres".replace("_", " "))

### 3.2 Vérifications avant l'entraînement

- Avec des poids aléatoires, le réseau ne préfère aucun style : la perte initiale doit être proche de ln 5.
- Un réseau sain doit savoir apprendre par cœur un seul batch. S'il n'y arrive pas, la boucle ou le modèle a un défaut.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

full_train_loader = DataLoader(datasets["train"], batch_size=len(datasets["train"]))
init_loss, _ = evaluate(model, full_train_loader, loss_fn, CPU)
print(f"Perte initiale sur le train : {init_loss:.3f}, attendue ≈ ln 5 = {np.log(len(CLASSES)):.3f}")

set_seed()
xb, yb = next(iter(make_train_loader()))
tiny = DenseNet(N_IN, HIDDEN, len(CLASSES))
optimizer = torch.optim.SGD(tiny.parameters(), lr=0.1)
for _ in range(300):
    loss = loss_fn(tiny(xb), yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
with torch.no_grad():
    batch_acc = (tiny(xb).argmax(1) == yb).float().mean().item()
print(f"Batch unique de {len(yb)} œuvres après 300 pas : perte {loss.item():.4f}, accuracy {batch_acc:.3f}")

### 3.3 Choix du taux d'apprentissage

SGD, batch de 64, 30 époques. Le taux d'apprentissage est choisi sur l'accuracy de validation à la dernière époque, en moyenne sur 3 graines : d'une graine à l'autre, elle varie de plusieurs points. La colonne `max_acc_val` donne le meilleur score atteint au fil des époques, à titre d'information seulement.

In [ ]:
N_EPOCHS = 30
LR_GRID = [0.01, 0.03, 0.1, 0.3]
SEEDS = [0, 1, 2]


def train_dense(lr: float, seed: int) -> tuple[DenseNet, dict[str, list[float]]]:
    """Entraîne le réseau de référence et renvoie le modèle et son historique par époque."""
    set_seed(seed)
    model = DenseNet(N_IN, HIDDEN, len(CLASSES))
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = fit(model, make_train_loader(), val_loader, loss_fn, optimizer, N_EPOCHS, CPU)
    return model, history


rows = []
for lr in LR_GRID:
    for seed in SEEDS:
        t0 = time.perf_counter()
        _, h = train_dense(lr, seed)
        rows.append({"lr": lr, "graine": seed, "acc_train": h["train_acc"][-1], "acc_val": h["val_acc"][-1],
                     "max_acc_val": max(h["val_acc"]), "entrainement_s": time.perf_counter() - t0})

runs = pd.DataFrame(rows)
lr_table = runs.groupby("lr")[["acc_train", "acc_val", "max_acc_val"]].agg(["mean", "std"])
lr_table.round(3)

### 3.4 Modèle retenu et comparaison avec la régression logistique

In [ ]:
best_lr = lr_table[("acc_val", "mean")].idxmax()
dense_runs = runs[runs["lr"] == best_lr]

dense, history = train_dense(best_lr, SEED)
y_pred = predict(dense, val_loader, CPU).numpy()
acc_dense = (y_pred == y_val).mean()
peak_epoch = int(np.argmax(history["val_acc"])) + 1
acc_logreg = acc_val  # section 2

print(f"lr retenu : {best_lr:g}")
print(f"Accuracy de validation sur {len(SEEDS)} graines : {dense_runs['acc_val'].mean():.3f} ± {dense_runs['acc_val'].std():.3f} "
      f"(de {dense_runs['acc_val'].min():.3f} à {dense_runs['acc_val'].max():.3f})")
print(f"Graine {SEED} : {acc_dense:.3f} en validation, {history['train_acc'][-1]:.3f} en train à la dernière époque ; "
      f"la validation culmine à {max(history['val_acc']):.3f} à l'époque {peak_epoch}")
print(f"Régression logistique (section 2) : {acc_logreg:.3f}")

### 3.5 Matrice de confusion

Sur la validation, pour le modèle de la graine 0 : une ligne par style réel, une colonne par style prédit.

In [ ]:
cm = pd.DataFrame(confusion_matrix(y_val, y_pred), index=CLASSES, columns=CLASSES)
cm.index.name, cm.columns.name = "style réel", "style prédit"

errors = cm.where(~np.eye(len(CLASSES), dtype=bool)).stack()
(true_style, pred_style), n_errors = errors.idxmax(), int(errors.max())
print(f"Erreur la plus fréquente : {true_style} pris pour {pred_style}, {n_errors} fois sur {cm.loc[true_style].sum()}")
print("Accuracy par style :", ", ".join(f"{s} {cm.loc[s, s] / cm.loc[s].sum():.2f}" for s in CLASSES))
cm

### 3.6 Sauvegarde

Le `state_dict` est sauvegardé dans `modeles/`, qui n'est pas versionné, puis rechargé dans un réseau neuf.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
dense_path = MODELS_DIR / "reseau_dense_baseline.pt"
torch.save(dense.state_dict(), dense_path)

reloaded = DenseNet(N_IN, HIDDEN, len(CLASSES))
reloaded.load_state_dict(torch.load(dense_path))
assert torch.equal(predict(reloaded, val_loader, CPU), predict(dense, val_loader, CPU))
print(f"{dense_path.name} : {dense_path.stat().st_size / 1e6:.1f} Mo, prédictions identiques après rechargement")

### 3.7 Bilan

- **Le réseau dense ne bat pas la régression logistique** : 0,302 ± 0,053 d'accuracy de validation sur 3 graines (de 0,243 à 0,347), contre 0,427. L'écart, d'environ 12 points, dépasse à la fois l'incertitude de mesure (± 5,6 points) et la variation d'une graine à l'autre.
- **Il sur-apprend** : 6,3 millions de paramètres pour 900 œuvres. À la dernière époque, la graine 0 atteint 0,850 en train mais 0,347 en validation, et la validation avait culminé à 0,360 dès l'époque 18. Sans régularisation, il fait à peine mieux que le hasard sur certaines graines, comme la régression logistique faiblement régularisée de la section 2 (0,350 à C = 1e-2).
- **Deux styles attirent les prédictions** : le Baroque (0,80 de bonnes réponses) et l'Ukiyo_e (0,55) sont reconnus, les trois autres presque jamais. L'erreur la plus fréquente est le Pop_Art pris pour de l'Ukiyo_e, 31 fois sur 60. Hypothèse à vérifier au jalon `m2` : sur des pixels bruts, le réseau s'appuie surtout sur les couleurs dominantes, sombres pour le Baroque, en aplats clairs pour l'Ukiyo_e comme pour le Pop_Art.
- **La boucle n'est pas en cause** : la perte initiale (1,646 pour ln 5 = 1,609) et l'apprentissage par cœur d'un batch le confirment. Le problème vient de la généralisation : le jalon `m2` le diagnostiquera sur les courbes de suivi et testera les corrections une par une.